# Notebook 01: Data Integration and Preprocessing
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Inspect the two raw datasets: `tinixai_it_ai_data_jobs.csv` and `topcv_jobs.csv`.
2. Standardize column schemas, assign global unique IDs (`unique_job_id`), and tag source provenance.
3. Perform intra-source and cross-source deduplication.
4. Clean text fields (stripping HTML, unescaping characters, preserving essential technical tokens like `C++`, `SQL`, `Python`, `AWS`).
5. Parse salary expressions into VND/month (`salary_min_vnd`, `salary_max_vnd`, `salary_mean_vnd`) with currency conversion and validation.
6. Generate standardized `job_role_label` categories using hierarchical keyword matching.
7. Export processed subsets for downstream classification, regression, and clustering.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.data.loader import load_raw_datasets, inspect_dataset, standardize_and_merge, deduplicate_merged
from src.data.cleaner import clean_dataframe
from src.data.salary_parser import process_salaries
from src.data.label_generator import generate_labels

## 1. Inspect Raw Datasets
Let's inspect row counts, schema definitions, and missing value profiles for both data sources without mutating raw files.

In [ ]:
df_tinixai, df_topcv = load_raw_datasets()

info_tinixai = inspect_dataset(df_tinixai, "TinixAI IT/AI Jobs")
info_topcv = inspect_dataset(df_topcv, "TopCV IT Jobs")

print("=== TinixAI Dataset ===")
print(f"Shape: {info_tinixai['rows']} rows x {info_tinixai['cols']} columns")
print(f"Missing values:\n{pd.Series(info_tinixai['missing_values'])}")

print("\n=== TopCV Dataset ===")
print(f"Shape: {info_topcv['rows']} rows x {info_topcv['cols']} columns")
print(f"Missing values:\n{pd.Series(info_topcv['missing_values'])}")

## 2. Standardize Schema and Deduplicate
We map both datasets into a common 16-attribute schema, synthesize `unique_job_id`, and check for duplicates within and across platforms.

In [ ]:
merged_raw = standardize_and_merge(df_tinixai, df_topcv)
print(f"Raw merged records: {len(merged_raw)}")

merged_deduped, removed = deduplicate_merged(merged_raw)
print(f"Records after deduplication: {len(merged_deduped)} (removed {len(removed)} duplicate postings)")

merged_deduped[['unique_job_id', 'source', 'job_title', 'company_name', 'salary', 'location']].head(5)

## 3. Text Cleaning & Normalization
Apply `clean_dataframe` to strip HTML tags, normalize whitespace, and preserve tech terms (e.g., C++, .NET, Python, SQL). Also normalize structured features: location regions, experience years, job position hierarchy, and year.

In [ ]:
cleaned_df = clean_dataframe(merged_deduped)
print("Sample cleaned text snippet:")
print(cleaned_df[['job_title', 'norm_location', 'exp_years', 'norm_experience', 'year_clean']].head(5))

## 4. Salary Parsing and Auditing
Convert salary strings into `salary_min_vnd`, `salary_max_vnd`, and `salary_mean_vnd`. Apply USD conversion (1 USD = 25,000 VND), parse millions/triệu, and filter implausible bounds (3M - 150M VND/month).

In [ ]:
parsed_df, salary_audit = process_salaries(cleaned_df)
print("=== Salary Parsing Audit Status ===")
print(salary_audit.to_string(index=False))

valid_salaries = parsed_df[parsed_df['salary_parse_status'] == 'valid']['salary_mean_vnd'] / 1_000_000
print("\nValid Salary Statistics (Million VND/month):")
print(valid_salaries.describe().round(2))

## 5. Job Role Label Generation
Apply hierarchical regex rules to `job_title` to assign ground truth roles for classification. Note: `job_title` is strictly withheld from the classification feature set.

In [ ]:
labeled_df, label_dist, label_mapping = generate_labels(parsed_df)
print("=== Job Role Label Distribution ===")
print(label_dist.to_string(index=False))

## 6. Export Task-Specific Subsets
Save the finalized data files to `data/processed/` for downstream modeling:
- `merged_jobs.csv`: Integrated raw jobs.
- `cleaned_jobs.csv`: Master cleaned dataset.
- `classification_data.csv`: Valid job role labels.
- `regression_data.csv`: Valid numeric salaries.
- `clustering_data.csv`: Valid composite texts.

In [ ]:
# Save master files
labeled_df.to_csv(config.PROCESSED_DATA_DIR / "cleaned_jobs.csv", index=False)

# Classification subset
cls_df = labeled_df[(labeled_df['job_role_label'] != 'Other / Unassigned') & (labeled_df['job_description'].str.strip() != '')]
cls_df.to_csv(config.PROCESSED_DATA_DIR / "classification_data.csv", index=False)

# Regression subset
reg_df = labeled_df[(labeled_df['salary_parse_status'] == 'valid') & (labeled_df['job_description'].str.strip() != '')]
reg_df.to_csv(config.PROCESSED_DATA_DIR / "regression_data.csv", index=False)

# Clustering subset
cl_df = labeled_df[labeled_df['job_description'].str.strip() != '']
cl_df.to_csv(config.PROCESSED_DATA_DIR / "clustering_data.csv", index=False)

print(f"Exported:\n- Cleaned: {len(labeled_df)}\n- Classification: {len(cls_df)}\n- Regression: {len(reg_df)}\n- Clustering: {len(cl_df)}")